# Non-overlapping Transformer forecasts on **test.csv**

Set **`SPLIT_DATA_DIR`** to the folder that contains **`train.csv`**, **`val.csv`**, and **`test.csv`**. The notebook loads **train** for normalization, **val** is loaded for parity (unused in plots), and **test** is the series you roll over and plot.

Raw splits are not smoothed on disk. If you trained with `--target-smoothing-window N`, set **`TARGET_SMOOTHING_WINDOW = N`** here (centered MA on `TARGET_COLUMN`, per file). Use **`1`** if the CSV values are already smoothed.

Adjust **`INPUT_WINDOW_LEN`**, **`FORECAST_LEN`**, **`WINDOW_STRIDE`** to match your checkpoint (e.g. non-overlapping: `stride == forecast_len`).


In [7]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from forecast_sweep_common import compute_uniform_timestep_start_indices, parse_timestamp_series
from forecast_sweep_common import smooth_target_series_1d
from train_transformer_sweep import TransformerForecastDelta

torch.set_grad_enabled(False)


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [ ]:
# Required: trained .pth checkpoint
MODEL_PATH = r""

# Optional: best_config.json (fills column names, smoothing, and CSV paths if present)
BEST_CONFIG_PATH = r""

# Folder containing train.csv, val.csv, test.csv
SPLIT_DATA_DIR = Path("data/splits/AHU_2_9_Blower_DE_A_r0.6_0.2_0.2")

# Plot title (empty -> auto from test file path)
PLOT_DATA_LABEL = ""

TARGET_COLUMN = "Acceleration RMS"
FEATURE_COLUMNS = ["Acceleration RMS"]
TARGET_SMOOTHING_WINDOW = 200
REQUIRE_UNIFORM_TIMESTEP = True
UNIFORM_STEP_SECONDS = 300
UNIFORM_TOL_SECONDS = 30
FILTER_TO_INCOMING_5MIN_ROWS = True
INCOMING_STEP_SECONDS = 300.0
INCOMING_STEP_TOL_SECONDS = 30.0

INPUT_WINDOW_LEN = 100
FORECAST_LEN = 100
WINDOW_STRIDE = 100
MAX_WINDOWS = None
SKIP_NON_UNIFORM_WINDOWS = True

PLOT_GAP_BREAK_SECONDS = 5 * 60
PLOT_FULL_TEST_GROUND_TRUTH = False

OUTPUT_DIR = Path("rolling_nonoverlap_transformer_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def existing_path_or_none(path_like):
    raw = str(path_like).strip()
    if not raw:
        return None
    path = Path(raw)
    return path if path.exists() else None


cfg_path = existing_path_or_none(BEST_CONFIG_PATH)
run_cfg = {}
if cfg_path is not None:
    run_cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    TARGET_COLUMN = run_cfg.get("value_column") or TARGET_COLUMN
    FEATURE_COLUMNS = run_cfg.get("feature_columns") or FEATURE_COLUMNS
    TARGET_SMOOTHING_WINDOW = int(run_cfg.get("target_smoothing_window") or TARGET_SMOOTHING_WINDOW)
    REQUIRE_UNIFORM_TIMESTEP = bool(run_cfg.get("require_uniform_timestep", REQUIRE_UNIFORM_TIMESTEP))
    UNIFORM_STEP_SECONDS = float(run_cfg.get("uniform_step_seconds") or UNIFORM_STEP_SECONDS)
    UNIFORM_TOL_SECONDS = float(run_cfg.get("uniform_step_tolerance_seconds") or UNIFORM_TOL_SECONDS)
    FILTER_TO_INCOMING_5MIN_ROWS = bool(run_cfg.get("filter_to_incoming_5min_rows", FILTER_TO_INCOMING_5MIN_ROWS))
    INCOMING_STEP_SECONDS = float(run_cfg.get("incoming_step_seconds") or INCOMING_STEP_SECONDS)
    INCOMING_STEP_TOL_SECONDS = float(run_cfg.get("incoming_step_tol_seconds") or INCOMING_STEP_TOL_SECONDS)

model_path = existing_path_or_none(MODEL_PATH)
if model_path is None or not model_path.is_file():
    raise FileNotFoundError("Set MODEL_PATH to your trained .pth checkpoint before running the notebook.")

split_root = Path(SPLIT_DATA_DIR).expanduser().resolve()
TRAIN_DATA_PATH = split_root / "train.csv"
VAL_DATA_PATH = split_root / "val.csv"
TEST_DATA_PATH = split_root / "test.csv"
if run_cfg.get("data_split_mode") == "explicit_train_val_test":
    tc = run_cfg.get("train_csv")
    vc = run_cfg.get("val_csv")
    te = run_cfg.get("test_csv")
    if tc and vc and te:
        TRAIN_DATA_PATH = Path(tc).expanduser().resolve()
        VAL_DATA_PATH = Path(vc).expanduser().resolve()
        TEST_DATA_PATH = Path(te).expanduser().resolve()

for label, p in (("train", TRAIN_DATA_PATH), ("val", VAL_DATA_PATH), ("test", TEST_DATA_PATH)):
    if not p.is_file():
        raise FileNotFoundError(f"{label} CSV not found: {p}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(model_path, map_location=device, weights_only=False)
model_cfg = dict(checkpoint.get("model_config") or run_cfg.get("model_config") or {})
ckpt_input_len = int(checkpoint.get("input_len") or model_cfg.get("input_len"))
ckpt_pred_len = int(checkpoint.get("pred_len") or model_cfg.get("pred_len"))
input_dim = int(model_cfg.get("input_dim", len(FEATURE_COLUMNS)))

if INPUT_WINDOW_LEN > ckpt_input_len:
    raise ValueError(f"INPUT_WINDOW_LEN={INPUT_WINDOW_LEN} > checkpoint input_len={ckpt_input_len}")
if FORECAST_LEN > ckpt_pred_len:
    raise ValueError(f"FORECAST_LEN={FORECAST_LEN} > checkpoint pred_len={ckpt_pred_len}")
if input_dim != len(FEATURE_COLUMNS):
    raise ValueError(f"Checkpoint input_dim={input_dim}, but FEATURE_COLUMNS has {len(FEATURE_COLUMNS)} columns")

model = TransformerForecastDelta(
    seq_len=ckpt_input_len,
    input_dim=input_dim,
    pred_len=ckpt_pred_len,
    d_model=int(model_cfg.get("d_model", 128)),
    nhead=int(model_cfg.get("nhead", 8)),
    num_layers=int(model_cfg.get("num_layers", 4)),
    dim_feedforward=int(model_cfg.get("dim_feedforward", 256)),
    dropout=float(model_cfg.get("dropout", 0.1)),
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded model on {device}: checkpoint input_len={ckpt_input_len}, pred_len={ckpt_pred_len}")
print(f"Using CSVs: {TRAIN_DATA_PATH}, {VAL_DATA_PATH}, {TEST_DATA_PATH}")


In [ ]:
def load_sort_parse(csv_path: Path) -> pd.DataFrame:
    frame = pd.read_csv(csv_path)
    frame = frame.copy()
    frame["TIMESTAMP"] = parse_timestamp_series(frame["TIMESTAMP"], str(csv_path))
    return frame.sort_values("TIMESTAMP").reset_index(drop=True)


def filter_incoming_5min(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    if not FILTER_TO_INCOMING_5MIN_ROWS:
        return frame
    ts = frame["TIMESTAMP"]
    dt_sec = ts.diff().dt.total_seconds().to_numpy()
    step_ok = np.zeros(len(ts), dtype=bool)
    step_ok[1:] = np.abs(dt_sec[1:] - float(INCOMING_STEP_SECONDS)) <= float(INCOMING_STEP_TOL_SECONDS)
    keep = np.zeros(len(ts), dtype=bool)
    if len(keep) > 0:
        keep[0] = len(ts) > 1 and bool(step_ok[1])
        if len(ts) > 1:
            keep[1:] = step_ok[1:]
    n_before = len(frame)
    out = frame.loc[keep].reset_index(drop=True)
    print(
        f"[{label}] FILTER_TO_INCOMING_5MIN_ROWS: kept {len(out)}/{n_before} rows "
        f"(dt from previous ~ {INCOMING_STEP_SECONDS}s +/- {INCOMING_STEP_TOL_SECONDS}s)."
    )
    return out


def smooth_value_column(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    w = int(TARGET_SMOOTHING_WINDOW)
    smooth_window = w if w % 2 == 1 else w + 1
    if smooth_window > 1:
        out[TARGET_COLUMN] = smooth_target_series_1d(
            out[TARGET_COLUMN].to_numpy(dtype=np.float32), smooth_window
        )
    return out


train_path = TRAIN_DATA_PATH
val_path = VAL_DATA_PATH
test_path = TEST_DATA_PATH

train_df = smooth_value_column(filter_incoming_5min(load_sort_parse(train_path), "train"))
_ = smooth_value_column(filter_incoming_5min(load_sort_parse(val_path), "val"))
df = smooth_value_column(filter_incoming_5min(load_sort_parse(test_path), "test"))

missing_cols = [c for c in [TARGET_COLUMN, *FEATURE_COLUMNS] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in test CSV {test_path}: {missing_cols}")

target = df[TARGET_COLUMN].to_numpy(dtype=np.float32)
features = df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
train_target = train_df[TARGET_COLUMN].to_numpy(dtype=np.float32)
train_features = train_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)

split_span = ckpt_input_len + ckpt_pred_len
if REQUIRE_UNIFORM_TIMESTEP:
    valid_starts = compute_uniform_timestep_start_indices(
        df["TIMESTAMP"], split_span, UNIFORM_STEP_SECONDS, UNIFORM_TOL_SECONDS
    )
else:
    valid_starts = np.arange(max(0, len(df) - split_span + 1), dtype=np.int64)

n = len(df)
plot_roll_start_row = 0
first_test_start_row = 0
test_overlay_begin_row = 0

train_mean = float(checkpoint.get("train_mean", float(np.mean(train_target))))
train_std = float(checkpoint.get("train_std", float(np.std(train_target) + 1e-8)))
feat_mean = train_features.mean(axis=0)
feat_std = train_features.std(axis=0) + 1e-8

auto_plot = f"{test_path.name} ({test_path.parent.name})"
PLOT_TITLE_NAME = (PLOT_DATA_LABEL or "").strip() or auto_plot

print(f"Loaded train {len(train_df)} rows: {train_path}")
print(f"Loaded test  {n} rows (rolling plot): {test_path}")
print(f"Normalization: train_mean={train_mean:.6f}, train_std={train_std:.6f} (checkpoint or train CSV).")

roll_span = int(INPUT_WINDOW_LEN + FORECAST_LEN)
if roll_span != split_span:
    print(
        f"Note: ckpt split_span={split_span} vs INPUT_WINDOW_LEN+FORECAST_LEN={roll_span}."
    )

print(
    f"Uniform span for valid_starts diag: split_span={split_span} @ {UNIFORM_STEP_SECONDS}s +/- {UNIFORM_TOL_SECONDS}s"
)
suffix = " + 5-min row filter" if FILTER_TO_INCOMING_5MIN_ROWS else ""
print(
    f"Test series (after sort{suffix}): rows 0..{n - 1}, "
    f"time {df['TIMESTAMP'].iloc[0]} -> {df['TIMESTAMP'].iloc[-1]}"
)
last_any_uniform_start = int(valid_starts[-1]) if len(valid_starts) else -1
if last_any_uniform_start >= 0:
    last_any_uniform_end = last_any_uniform_start + split_span - 1
    if last_any_uniform_end < n - 1:
        print(
            f"Trailing rows after last uniform window end ({df['TIMESTAMP'].iloc[last_any_uniform_end]}): "
            f"{n - 1 - last_any_uniform_end} row(s)."
        )


In [ ]:
def window_is_uniform(start, length):
    if not SKIP_NON_UNIFORM_WINDOWS or length <= 1:
        return True
    ts = df["TIMESTAMP"].iloc[start:start + length].to_numpy(dtype="datetime64[ns]").astype("int64")
    diffs = np.diff(ts) / 1e9
    return bool(np.all(np.abs(diffs - UNIFORM_STEP_SECONDS) <= UNIFORM_TOL_SECONDS))


def predict_from_ground_truth_window(start):
    x_raw = features[start:start + INPUT_WINDOW_LEN]
    x_norm = (x_raw - feat_mean) / feat_std
    x_tensor = torch.tensor(x_norm.T[None, :, :], dtype=torch.float32, device=device)
    last_val_norm = (target[start + INPUT_WINDOW_LEN - 1] - train_mean) / train_std
    pred_delta = model(x_tensor).detach().cpu().numpy()[0]
    pred_abs_norm = pred_delta + float(last_val_norm)
    return (pred_abs_norm * train_std + train_mean)[:FORECAST_LEN]


rows = []
n_skip_uniform = 0
first_used_start = None
first_pred_arr = None
per_window_dir = OUTPUT_DIR / "per_window_predictions"
per_window_dir.mkdir(parents=True, exist_ok=True)
max_start = len(df) - INPUT_WINDOW_LEN - FORECAST_LEN
upper_start = int(max_start)

if plot_roll_start_row > upper_start:
    raise ValueError(
        f"No rolling candidates: plot_roll_start_row={plot_roll_start_row} > upper_start={upper_start}. "
        "Test region may be shorter than one input+forecast span; shorten INPUT_WINDOW_LEN/FORECAST_LEN or use more rows."
    )
candidate_starts = np.arange(plot_roll_start_row, upper_start + 1, WINDOW_STRIDE, dtype=np.int64)
if MAX_WINDOWS is not None:
    candidate_starts = candidate_starts[:int(MAX_WINDOWS)]

for source_idx, start in enumerate(candidate_starts):
    if not window_is_uniform(int(start), INPUT_WINDOW_LEN + FORECAST_LEN):
        n_skip_uniform += 1
        print(f"Skipping source window {source_idx} at row {start}: non-uniform timestamps")
        continue
    pred = predict_from_ground_truth_window(int(start))
    if first_used_start is None:
        first_used_start = int(start)
        first_pred_arr = np.asarray(pred, dtype=np.float32).copy()
    forecast_start = int(start) + INPUT_WINDOW_LEN
    forecast_rows = np.arange(forecast_start, forecast_start + FORECAST_LEN, dtype=np.int64)
    window_df = pd.DataFrame({
        "source_window_index": source_idx,
        "source_input_start_row": int(start),
        "forecast_window_index": source_idx + 1,
        "row_index": forecast_rows,
        "step_ahead": np.arange(1, FORECAST_LEN + 1),
        "timestamp": df["TIMESTAMP"].iloc[forecast_rows].astype(str).to_list(),
        "predicted": pred,
        "actual": target[forecast_rows],
    })
    window_df.to_csv(per_window_dir / f"prediction_window_{source_idx + 1:04d}.csv", index=False)
    rows.append(window_df)

if not rows:
    raise ValueError("No rolling prediction windows were produced.")
combined = pd.concat(rows, ignore_index=True)
combined_path = OUTPUT_DIR / "combined_nonoverlap_predictions.csv"
combined.to_csv(combined_path, index=False)

if first_used_start is None:
    raise RuntimeError("first_used_start unset despite non-empty predictions.")
input_rows = np.arange(first_used_start, first_used_start + INPUT_WINDOW_LEN, dtype=np.int64)
first_input = pd.DataFrame({
    "row_index": input_rows,
    "timestamp": df["TIMESTAMP"].iloc[input_rows].astype(str).to_list(),
    "smoothed_gt_input": target[input_rows],
})
first_input_path = OUTPUT_DIR / "first_input_ground_truth_window.csv"
first_input.to_csv(first_input_path, index=False)

fo0 = first_used_start + INPUT_WINDOW_LEN
forecast_rows_0 = np.arange(fo0, fo0 + FORECAST_LEN, dtype=np.int64)
first_window_forecast = pd.DataFrame({
    "row_index": forecast_rows_0,
    "timestamp": df["TIMESTAMP"].iloc[forecast_rows_0].astype(str).to_list(),
    "smoothed_gt_output": target[forecast_rows_0],
    "predicted": first_pred_arr,
})
first_window_forecast.to_csv(
    OUTPUT_DIR / "first_window_output_predicted_vs_smoothed_gt.csv", index=False
)

if PLOT_FULL_TEST_GROUND_TRUTH:
    t_ov = int(test_overlay_begin_row)
    plot_test_full = pd.DataFrame(
        {
            "timestamp": pd.to_datetime(df["TIMESTAMP"].iloc[t_ov:].reset_index(drop=True)),
            TARGET_COLUMN: target[t_ov:],
        }
    )
else:
    plot_test_full = None

t_forecast = pd.to_datetime(combined["timestamp"])
t_end_data = df["TIMESTAMP"].iloc[-1]
if t_forecast.max() < t_end_data:
    print(
        f"Note: rolling forecast/actual points end at {t_forecast.max()} but data ends at {t_end_data}. "
        "The test period still includes later rows; enable PLOT_FULL_TEST_GROUND_TRUTH to draw them."
    )

ts_plot_min = min(
    pd.to_datetime(first_input["timestamp"]).min(),
    pd.to_datetime(first_window_forecast["timestamp"]).min(),
    pd.to_datetime(combined["timestamp"]).min(),
)
ts_plot_max = max(
    pd.to_datetime(first_input["timestamp"]).max(),
    pd.to_datetime(first_window_forecast["timestamp"]).max(),
    pd.to_datetime(combined["timestamp"]).max(),
)
print(
    f"Rolling plot timestamp range: {ts_plot_min} → {ts_plot_max} "
    f"(vs CSV {df['TIMESTAMP'].iloc[0]} → {df['TIMESTAMP'].iloc[-1]})"
)
print(
    f"Candidates: {len(candidate_starts)}, windows saved: {len(rows)}, "
    f"skipped (non-uniform over roll span): {n_skip_uniform}. "
    "Large skips → disjoint blobs on the x-axis; that is expected."
)
print(f"Saved {len(rows)} prediction windows to {per_window_dir}")
print(
    f"First used rolling window: start row {first_used_start} "
    f"({df['TIMESTAMP'].iloc[first_used_start]}); "
    f"input rows [{first_used_start}, {first_used_start + INPUT_WINDOW_LEN}), "
    f"output rows [{first_used_start + INPUT_WINDOW_LEN}, {first_used_start + INPUT_WINDOW_LEN + FORECAST_LEN})."
)


In [ ]:
plot_input = first_input.copy()
plot_pred = combined.copy()
plot_first_fc = first_window_forecast.copy()
plot_input["timestamp"] = pd.to_datetime(plot_input["timestamp"])
plot_pred["timestamp"] = pd.to_datetime(plot_pred["timestamp"])
plot_first_fc["timestamp"] = pd.to_datetime(plot_first_fc["timestamp"])


def plot_with_gap_breaks(ax, frame, x_col, y_col, *, max_gap_seconds, label, **plot_kwargs):
    frame = frame.sort_values(x_col).reset_index(drop=True)
    if frame.empty:
        return

    gaps = frame[x_col].diff().dt.total_seconds().gt(max_gap_seconds).fillna(False)
    segment_ids = gaps.cumsum()
    first_segment = True
    for _, segment in frame.groupby(segment_ids, sort=False):
        ax.plot(
            segment[x_col],
            segment[y_col],
            label=label if first_segment else None,
            **plot_kwargs,
        )
        first_segment = False


fig, ax = plt.subplots(figsize=(18, 5))
if plot_test_full is not None and not plot_test_full.empty:
    plot_with_gap_breaks(
        ax,
        plot_test_full,
        "timestamp",
        TARGET_COLUMN,
        max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
        color="0.65",
        linewidth=0.9,
        alpha=0.45,
        zorder=1,
        label="Full test row-range — smoothed GT (background)",
    )
# All rolling output windows (lighter)
plot_with_gap_breaks(
    ax,
    plot_pred,
    "timestamp",
    "actual",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
    color="C0",
    linewidth=1.0,
    alpha=0.5,
    zorder=2,
    label=f"Output — smoothed GT, all windows ({FORECAST_LEN} steps; stride={WINDOW_STRIDE})",
)
plot_with_gap_breaks(
    ax,
    plot_pred,
    "timestamp",
    "predicted",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
    color="C1",
    linewidth=1.0,
    alpha=0.5,
    zorder=2,
    label="Output — predicted, all windows",
)
# First window: input + output (emphasized)
plot_with_gap_breaks(
    ax,
    plot_input,
    "timestamp",
    "smoothed_gt_input",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
    color="0.12",
    linewidth=2.1,
    zorder=4,
    label=f"Input — 1st window smoothed GT ({INPUT_WINDOW_LEN})",
)
plot_with_gap_breaks(
    ax,
    plot_first_fc,
    "timestamp",
    "smoothed_gt_output",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
    color="C0",
    linewidth=2.2,
    alpha=0.95,
    zorder=5,
    label=f"Output — 1st window smoothed GT ({FORECAST_LEN})",
)
plot_with_gap_breaks(
    ax,
    plot_first_fc,
    "timestamp",
    "predicted",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
    color="C1",
    linewidth=2.2,
    alpha=0.95,
    zorder=5,
    label=f"Output — 1st window predicted ({FORECAST_LEN})",
)
t_in_end = plot_input["timestamp"].iloc[-1]
t_out_end = plot_first_fc["timestamp"].iloc[-1]
ax.axvline(t_in_end, color="0.45", linestyle="--", linewidth=1.0, zorder=3)
ax.axvline(t_out_end, color="0.55", linestyle=":", linewidth=1.0, zorder=3)
ax.set_title(
    f"{PLOT_TITLE_NAME}: input vs output — smoothed GT vs predicted "
    f"(input_len={INPUT_WINDOW_LEN}, forecast_len={FORECAST_LEN}, stride={WINDOW_STRIDE})"
)
ax.set_xlabel("Timestamp")
ax.set_ylabel(TARGET_COLUMN)
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
fig.autofmt_xdate()
fig.tight_layout()
plot_path = OUTPUT_DIR / "first_input_then_ground_truth_vs_predictions.png"
fig.savefig(plot_path, dpi=150)
print(f"Saved plot to {plot_path}")


In [ ]:
# Interactive zoomable version of the same plot.
# If this import fails, install Plotly in the notebook kernel: pip install plotly
import plotly.graph_objects as go


def trace_arrays_with_gap_breaks(frame, x_col, y_col, *, max_gap_seconds):
    frame = frame.sort_values(x_col).reset_index(drop=True)
    xs = []
    ys = []
    for idx, row in frame.iterrows():
        if idx > 0:
            gap_seconds = (row[x_col] - frame.loc[idx - 1, x_col]).total_seconds()
            if gap_seconds > max_gap_seconds:
                xs.append(None)
                ys.append(None)
        xs.append(row[x_col])
        ys.append(row[y_col])
    return xs, ys


fig = go.Figure()
x_full, y_full = (None, None)
if plot_test_full is not None and not plot_test_full.empty:
    x_full, y_full = trace_arrays_with_gap_breaks(
        plot_test_full,
        "timestamp",
        TARGET_COLUMN,
        max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
    )
    fig.add_trace(
        go.Scatter(
            x=x_full,
            y=y_full,
            mode="lines",
            name="Test range — smoothed GT (background)",
            line=dict(color="rgba(160,160,160,0.5)", width=1),
        )
    )

x_act_all, y_act_all = trace_arrays_with_gap_breaks(
    plot_pred,
    "timestamp",
    "actual",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
)
x_pred_all, y_pred_all = trace_arrays_with_gap_breaks(
    plot_pred,
    "timestamp",
    "predicted",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
)
fig.add_trace(
    go.Scatter(
        x=x_act_all,
        y=y_act_all,
        mode="lines",
        name=f"Output — smoothed GT, all windows ({FORECAST_LEN} steps, stride={WINDOW_STRIDE})",
        line=dict(color="rgba(31,119,180,0.5)", width=1.3),
    )
)
fig.add_trace(
    go.Scatter(
        x=x_pred_all,
        y=y_pred_all,
        mode="lines",
        name="Output — predicted, all windows",
        line=dict(color="rgba(255,127,14,0.5)", width=1.3),
    )
)

x_input, y_input = trace_arrays_with_gap_breaks(
    plot_input,
    "timestamp",
    "smoothed_gt_input",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
)
x_fc_gt, y_fc_gt = trace_arrays_with_gap_breaks(
    plot_first_fc,
    "timestamp",
    "smoothed_gt_output",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
)
x_fc_pr, y_fc_pr = trace_arrays_with_gap_breaks(
    plot_first_fc,
    "timestamp",
    "predicted",
    max_gap_seconds=PLOT_GAP_BREAK_SECONDS,
)
fig.add_trace(
    go.Scatter(
        x=x_input,
        y=y_input,
        mode="lines",
        name=f"Input — 1st window smoothed GT ({INPUT_WINDOW_LEN})",
        line=dict(color="rgba(30,30,30,1)", width=2.4),
    )
)
fig.add_trace(
    go.Scatter(
        x=x_fc_gt,
        y=y_fc_gt,
        mode="lines",
        name=f"Output — 1st window smoothed GT ({FORECAST_LEN})",
        line=dict(color="rgba(31,119,180,1)", width=2.2),
    )
)
fig.add_trace(
    go.Scatter(
        x=x_fc_pr,
        y=y_fc_pr,
        mode="lines",
        name=f"Output — 1st window predicted ({FORECAST_LEN})",
        line=dict(color="rgba(255,127,14,1)", width=2.2),
    )
)
fig.add_vline(
    x=plot_input["timestamp"].iloc[-1],
    line_dash="dash",
    line_color="gray",
    line_width=1,
)
fig.add_vline(
    x=plot_first_fc["timestamp"].iloc[-1],
    line_dash="dot",
    line_color="rgba(100,100,100,0.8)",
    line_width=1,
)
fig.update_layout(
    title=(
        f"Interactive: {PLOT_TITLE_NAME} — input vs output (smoothed GT vs predicted); "
        f"input_len={INPUT_WINDOW_LEN}, forecast_len={FORECAST_LEN}, stride={WINDOW_STRIDE}"
    ),
    xaxis_title="Timestamp",
    yaxis_title=TARGET_COLUMN,
    hovermode="x unified",
    template="plotly_white",
    width=1200,
    height=520,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.update_xaxes(rangeslider_visible=True)

interactive_plot_path = OUTPUT_DIR / "interactive_ground_truth_vs_predictions.html"
fig.write_html(interactive_plot_path)
print(f"Saved interactive plot to {interactive_plot_path}")
fig.show()
